In [5]:
from bs4 import BeautifulSoup
import json
import re

In [8]:
def read_html(url):
    with open(url, encoding="utf-8") as f:
        content = f.read()
        soup = BeautifulSoup(content)
    return soup

In [91]:
def get_verses(soup):

    docs = []
    chunk_id = 0

    testament, book = [None] * 2

    for ch in soup.select(".chapter"):

        h2 = ch.find_all("h2")

        if h2:

            if len(h2) == 1:
                
                heading = h2[0].get_text()

                if "Testament" in heading:
                    testament = heading
                else:
                    book = heading
            else:
                raise AssertionError


        paras = ch.find_all("p")

        prev_chunk, curr_chunk = [None] * 2

        prev_beg, prev_end = [None] * 2
        curr_beg, curr_end = [None] * 2

        for p in paras:


            chunk = p.get_text()
            numbers = re.findall(r"\d+:\d+", chunk)

            prev_chunk = curr_chunk
            curr_chunk = chunk

            if len(numbers) >= 1:

                prev_beg = curr_beg
                curr_beg = numbers[0]

                prev_end = curr_end
                curr_end = numbers[-1]
            
            else:

                if prev_chunk != None:
                    
                    if any(x in prev_chunk for x in ["\nCommonly Called:\n", "\nOtherwise Called:\n", "\nor\n"]):
                        book += f"({curr_chunk})"
                        continue

                    else:
                        chunk = prev_chunk + curr_chunk

                        # change the previous document instead of creating a new documnet
                        docs[-1]["text"] = chunk
                        docs[-1]["metadata"]["beg_verse"] = prev_beg
                        docs[-1]["metadata"]["end_verse"] = prev_end
                        
                        continue


            chunk_id += 1

            doc = {
                "text": chunk,
                "metadata": {
                    "chunk_id": chunk_id,
                    "testament": testament,
                    "book": book,
                    "beg_verse": curr_beg,
                    "end_verse": curr_end
                }  
            }


            docs += [doc]

    return docs


In [92]:
def main():
    soup = read_html("../../data/raw/bible.html")
    docs = get_verses(soup)

    with open("../../data/processed/bible.json", "w", encoding="utf-8") as f:
        json.dump(docs, f, indent=4) 
    

In [93]:
main()